In [2]:
# inlegalbert_bilstm_crf_rrc.py

import os, json, random, time
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModel,
    get_linear_schedule_with_warmup,
)
from torchcrf import CRF
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score,
    precision_score, recall_score,
)

# ═══════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════
INLEGALBERT_MODEL_NAME = "law-ai/InLegalBERT"
TRAIN_PATH = "dataset/build_train.jsonl"
DEV_PATH   = "dataset/build_dev.jsonl"
TEST_PATH  = "dataset/build_test.jsonl"
OUT_DIR    = "rrc_bilstm_crf_logs"
BEST_MODEL_DIR = os.path.join(OUT_DIR, "best_model")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(BEST_MODEL_DIR, exist_ok=True)

SEED            = 42
MAX_SEQ_LENGTH  = 32
BATCH_DOCS      = 2
NUM_EPOCHS      = 60
BERT_LR         = 2e-5
HEAD_LR         = 1e-3
WEIGHT_DECAY    = 0.01
GRAD_CLIP       = 1.0
DROPOUT         = 0.3
LSTM_HIDDEN     = 16        # per direction; output = 32
LSTM_LAYERS     = 2
WARMUP_RATIO    = 0.05
GRADIENT_ACCUMULATION_STEPS = 2

# Rare-class threshold: any label whose share of total train labels
# is ≤ RARE_THRESHOLD is treated as a rare class for separate reporting.
RARE_THRESHOLD = 0.05        # 5 %

LABELS = [
    "PREAMBLE", "FAC", "RLC", "ISSUE", "ARG_PETITIONER",
    "ARG_RESPONDENT", "ANALYSIS", "STA", "PRE_RELIED",
    "PRE_NOT_RELIED", "RATIO", "RPC", "NONE",
]
label2id   = {lbl: i for i, lbl in enumerate(LABELS)}
id2label   = {i: lbl for i, lbl in enumerate(LABELS)}
NUM_LABELS = len(LABELS)
DEVICE     = "cuda:0"

# ═══════════════════════════════════════════════════════════
# SEED
# ═══════════════════════════════════════════════════════════
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()


# ═══════════════════════════════════════════════════════════
# PARAMETER COUNTER  ← NEW
# ═══════════════════════════════════════════════════════════
def count_parameters(model):
    """
    Returns a detailed breakdown of trainable and frozen parameters.

    Returns:
        total_trainable  (int)
        total_frozen     (int)
        component_table  (list[dict])  — per-component breakdown
    """
    component_map = {
        "bert":       "InLegalBERT Encoder",
        "bilstm":     "BiLSTM",
        "classifier": "Classifier Head",
        "crf":        "CRF",
        "dropout":    "Dropout",
    }

    rows = []
    for attr, name in component_map.items():
        module = getattr(model, attr, None)
        if module is None:
            continue
        trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
        frozen    = sum(p.numel() for p in module.parameters() if not p.requires_grad)
        rows.append({
            "Component":         name,
            "Trainable Params":  trainable,
            "Frozen Params":     frozen,
            "Total Params":      trainable + frozen,
        })

    total_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    total_all       = total_trainable + total_frozen

    rows.append({
        "Component":        "── TOTAL ──",
        "Trainable Params": total_trainable,
        "Frozen Params":    total_frozen,
        "Total Params":     total_all,
    })

    print("\n" + "=" * 66)
    print("MODEL PARAMETER SUMMARY")
    print("=" * 66)
    print(f"  {'Component':<25} {'Trainable':>14} {'Frozen':>10} {'Total':>12}")
    print("-" * 66)
    for r in rows:
        sep = "─" * 66 if r["Component"] == "── TOTAL ──" else ""
        if sep: print(sep)
        print(
            f"  {r['Component']:<25} "
            f"{r['Trainable Params']:>14,} "
            f"{r['Frozen Params']:>10,} "
            f"{r['Total Params']:>12,}"
        )
    print("=" * 66)

    return total_trainable, total_frozen, rows


# ═══════════════════════════════════════════════════════════
# RARE-CLASS AUTO-DETECTION
# ═══════════════════════════════════════════════════════════
def detect_rare_classes(docs, threshold=RARE_THRESHOLD):
    """
    Compute the frequency of every label in `docs` and return
    the list of label-strings whose share ≤ threshold.
    """
    all_ids = [lid for _, labs in docs for lid in labs]
    total   = len(all_ids)
    counts  = Counter(all_ids)
    label_freqs = {id2label[i]: counts.get(i, 0) / total for i in range(NUM_LABELS)}

    rare_labels = [id2label[i] for i in range(NUM_LABELS)
                   if label_freqs[id2label[i]] <= threshold]
    rare_ids    = [label2id[l] for l in rare_labels]

    print(f"\n📊 Label frequency analysis (threshold ≤ {threshold*100:.0f}%):")
    for lbl in LABELS:
        freq  = label_freqs[lbl]
        flag  = " ← RARE" if lbl in rare_labels else ""
        count = counts.get(label2id[lbl], 0)
        print(f"   {lbl:<20} {freq*100:5.2f}%  ({count:5d} samples){flag}")
    print(f"\n   Rare classes ({len(rare_labels)}): {rare_labels}\n")

    return rare_labels, rare_ids, label_freqs


# ═══════════════════════════════════════════════════════════
# DATA LOADING
# ═══════════════════════════════════════════════════════════
def load_jsonl(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                data.append(json.loads(s))
    return data


def extract_docs(docs, max_sents=256):
    all_docs = []
    for doc in docs:
        sents, labs = [], []

        if "sentences" in doc and "labels" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["labels"]]
        elif "sentences" in doc and "annotation" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["annotation"]]
        elif "annotations" in doc:
            for a in doc.get("annotations", []):
                for item in a.get("result", []):
                    val  = item.get("value", {})
                    text = val.get("text", "").strip()
                    lbl  = val.get("labels", ["NONE"])[0]
                    if text:
                        sents.append(text)
                        labs.append(label2id.get(lbl, label2id["NONE"]))

        if not sents or len(sents) != len(labs):
            continue

        sents = sents[:max_sents]
        labs  = labs[:max_sents]
        all_docs.append((sents, labs))
    return all_docs


# ═══════════════════════════════════════════════════════════
# DATASET
# ═══════════════════════════════════════════════════════════
class RRCDataset(Dataset):
    def __init__(self, docs, tokenizer, max_length=MAX_SEQ_LENGTH):
        self.docs       = docs
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.docs)

    def __getitem__(self, idx):
        sents, labels = self.docs[idx]
        enc = self.tokenizer(
            sents,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "token_type_ids": enc.get(
                "token_type_ids",
                torch.zeros_like(enc["input_ids"])
            ),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


def collate_rrc(batch):
    T_max = max(b["input_ids"].shape[0] for b in batch)
    L     = batch[0]["input_ids"].shape[1]
    B     = len(batch)

    input_ids      = torch.zeros(B, T_max, L, dtype=torch.long)
    attention_mask = torch.zeros(B, T_max, L, dtype=torch.long)
    token_type_ids = torch.zeros(B, T_max, L, dtype=torch.long)
    labels         = torch.full((B, T_max), fill_value=-100, dtype=torch.long)
    lengths        = torch.zeros(B, dtype=torch.long)

    for i, b in enumerate(batch):
        t = b["input_ids"].shape[0]
        input_ids[i, :t]      = b["input_ids"]
        attention_mask[i, :t] = b["attention_mask"]
        token_type_ids[i, :t] = b["token_type_ids"]
        labels[i, :t]         = b["labels"]
        lengths[i]            = t

    return input_ids, attention_mask, token_type_ids, labels, lengths


# ═══════════════════════════════════════════════════════════
# MODEL
# ═══════════════════════════════════════════════════════════
class InLegalBERT_BiLSTM_CRF(nn.Module):
    """
    InLegalBERT (full fine-tune) → mean pool → BiLSTM → Linear → CRF
    """

    def __init__(
        self,
        bert_model_name=INLEGALBERT_MODEL_NAME,
        lstm_hidden=LSTM_HIDDEN,
        lstm_layers=LSTM_LAYERS,
        num_labels=NUM_LABELS,
        dropout=DROPOUT,
    ):
        super().__init__()
        self.bert     = AutoModel.from_pretrained(bert_model_name)
        self.bert_dim = self.bert.config.hidden_size   # 768

        self.dropout = nn.Dropout(dropout)
        self.bilstm  = nn.LSTM(
            input_size   = self.bert_dim,
            hidden_size  = lstm_hidden,
            num_layers   = lstm_layers,
            bidirectional=True,
            batch_first  =True,
            dropout=dropout if lstm_layers > 1 else 0.0,
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(lstm_hidden * 2, lstm_hidden),
            nn.Sigmoid(),
            nn.Dropout(dropout),
            nn.Linear(lstm_hidden, num_labels),
        )
        self.crf = CRF(num_tags=num_labels, batch_first=True)

    def encode_sentences(self, input_ids, attention_mask, token_type_ids):
        B, T, L   = input_ids.shape
        flat_ids  = input_ids.view(B * T, L)
        flat_mask = attention_mask.view(B * T, L)
        flat_types= token_type_ids.view(B * T, L)

        outputs   = self.bert(
            input_ids      =flat_ids,
            attention_mask =flat_mask,
            token_type_ids =flat_types,
        )
        hidden   = outputs.last_hidden_state           # (B*T, L, H)
        mask_f   = flat_mask.unsqueeze(-1).float()
        sent_emb = (hidden * mask_f).sum(1) / mask_f.sum(1).clamp(min=1e-9)
        return sent_emb.view(B, T, -1)                 # (B, T, H)

    def forward(self, input_ids, attention_mask, token_type_ids,
                labels=None, lengths=None):

        sent_emb = self.dropout(
            self.encode_sentences(input_ids, attention_mask, token_type_ids)
        )                                              # (B, T, bert_dim)

        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                sent_emb, lengths.cpu(), batch_first=True, enforce_sorted=False
            )
            packed_out, _ = self.bilstm(packed)
            lstm_out, _   = nn.utils.rnn.pad_packed_sequence(
                packed_out, batch_first=True
            )
        else:
            lstm_out, _ = self.bilstm(sent_emb)        # (B, T, 2H)

        emissions = self.classifier(lstm_out)           # (B, T, C)

        # CRF mask
        if lengths is not None:
            B, T, _ = emissions.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=emissions.device)
            for i, l in enumerate(lengths):
                mask[i, :l] = True
        elif labels is not None:
            mask = (labels != -100)
        else:
            mask = torch.ones(emissions.shape[:2], dtype=torch.bool,
                              device=emissions.device)

        if labels is not None:
            safe_labels = labels.clone()
            safe_labels[safe_labels == -100] = 0
            loss = -self.crf(emissions, safe_labels, mask=mask, reduction="mean")
            return loss, emissions
        else:
            return self.crf.decode(emissions, mask=mask), emissions


# ═══════════════════════════════════════════════════════════
# METRICS HELPER
# ═══════════════════════════════════════════════════════════
def compute_all_metrics(all_trues, all_preds, rare_ids, split_name=""):
    """
    Full metrics suite including per-class F1, precision, recall.

    Returns a dict with scalar aggregates + per-class breakdown + cm + report.
    """
    str_trues = [id2label[x] for x in all_trues]
    str_preds = [id2label[x] for x in all_preds]

    # ── Aggregate F1 ─────────────────────────────────────────
    macro_f1    = f1_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_f1    = f1_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_f1 = f1_score(all_trues, all_preds, average="weighted", zero_division=0)

    # ── Aggregate Precision ───────────────────────────────────
    macro_prec    = precision_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_prec    = precision_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_prec = precision_score(all_trues, all_preds, average="weighted", zero_division=0)

    # ── Aggregate Recall ──────────────────────────────────────
    macro_rec    = recall_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_rec    = recall_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_rec = recall_score(all_trues, all_preds, average="weighted", zero_division=0)

    # ── Accuracy ──────────────────────────────────────────────
    acc = accuracy_score(all_trues, all_preds)

    # ── Per-class F1, Precision, Recall  ← NEW ───────────────
    per_class_f1  = f1_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                             average=None, zero_division=0)
    per_class_prec = precision_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                                     average=None, zero_division=0)
    per_class_rec  = recall_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                                  average=None, zero_division=0)

    per_class_metrics = {
        id2label[i]: {
            "f1":        float(per_class_f1[i]),
            "precision": float(per_class_prec[i]),
            "recall":    float(per_class_rec[i]),
        }
        for i in range(NUM_LABELS)
    }

    # ── Minority / Rare-class metrics ─────────────────────────
    present_rare = [r for r in rare_ids if r in all_trues]
    if present_rare:
        rare_f1   = f1_score(all_trues, all_preds, labels=present_rare,
                             average="macro", zero_division=0)
        rare_prec = precision_score(all_trues, all_preds, labels=present_rare,
                                    average="macro", zero_division=0)
        rare_rec  = recall_score(all_trues, all_preds, labels=present_rare,
                                 average="macro", zero_division=0)
    else:
        rare_f1 = rare_prec = rare_rec = 0.0

    # ── Per-class report ──────────────────────────────────────
    cls_report = classification_report(
        str_trues, str_preds,
        labels=LABELS,
        digits=4,
        zero_division=0,
    )

    # ── Confusion matrix ──────────────────────────────────────
    cm = confusion_matrix(str_trues, str_preds, labels=LABELS)

    return {
        # F1
        "macro_f1":            macro_f1,
        "micro_f1":            micro_f1,
        "weighted_f1":         weighted_f1,
        # Precision
        "macro_precision":     macro_prec,
        "micro_precision":     micro_prec,
        "weighted_precision":  weighted_prec,
        # Recall
        "macro_recall":        macro_rec,
        "micro_recall":        micro_rec,
        "weighted_recall":     weighted_rec,
        # Rare / Minority
        "rare_f1":             rare_f1,
        "rare_precision":      rare_prec,
        "rare_recall":         rare_rec,
        # Per-class
        "per_class_metrics":   per_class_metrics,
        # Scalar
        "accuracy":            acc,
        # Reports
        "cls_report":          cls_report,
        "cm":                  cm,
        # Raw for further use
        "all_preds":           all_preds,
        "all_trues":           all_trues,
    }


# ═══════════════════════════════════════════════════════════
# TRAINER
# ═══════════════════════════════════════════════════════════
class Trainer:
    def __init__(self, model, device=DEVICE):
        self.model  = model.to(device)
        self.device = device

    # ── Two-LR optimiser ──────────────────────────────────────
    def build_optimizer(self):
        bert_params = list(self.model.bert.parameters())
        head_params = (
            list(self.model.bilstm.parameters())
            + list(self.model.classifier.parameters())
            + list(self.model.crf.parameters())
            + list(self.model.dropout.parameters())
        )
        return torch.optim.AdamW(
            [
                {"params": bert_params, "lr": BERT_LR},
                {"params": head_params, "lr": HEAD_LR},
            ],
            weight_decay=WEIGHT_DECAY,
        )

    # ── Validation loss ───────────────────────────────────────
    def compute_val_loss(self, dataset):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        total_loss, n = 0.0, 0
        with torch.no_grad():
            for input_ids, attention_mask, token_type_ids, labels, lengths in loader:
                input_ids      = input_ids.to(self.device)
                attention_mask = attention_mask.to(self.device)
                token_type_ids = token_type_ids.to(self.device)
                labels         = labels.to(self.device)
                lengths        = lengths.to(self.device)
                loss, _ = self.model(
                    input_ids, attention_mask, token_type_ids,
                    labels=labels, lengths=lengths,
                )
                total_loss += loss.item()
                n += 1
        return total_loss / max(1, n)

    # ── Main training loop ────────────────────────────────────
    def train(self, train_dataset, dev_dataset, rare_ids,
              tokenizer, num_epochs=NUM_EPOCHS):

        train_loader = DataLoader(
            train_dataset, batch_size=BATCH_DOCS,
            shuffle=True, collate_fn=collate_rrc,
        )
        optimizer    = self.build_optimizer()
        total_steps  = len(train_loader) * num_epochs // GRADIENT_ACCUMULATION_STEPS
        warmup_steps = int(WARMUP_RATIO * total_steps)
        scheduler    = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps =warmup_steps,
            num_training_steps=total_steps,
        )

        history    = []
        best_f1    = -1.0
        best_state = None

        # ── Total training timer  ← NEW ───────────────────────
        total_train_start = time.time()

        for epoch in range(1, num_epochs + 1):
            # ── TRAIN ──────────────────────────────────────────
            self.model.train()
            running_loss, n_steps = 0.0, 0
            epoch_start = time.time()                   # ← NEW
            optimizer.zero_grad()

            for step, (input_ids, attention_mask,
                        token_type_ids, labels, lengths) in enumerate(train_loader):

                input_ids      = input_ids.to(self.device)
                attention_mask = attention_mask.to(self.device)
                token_type_ids = token_type_ids.to(self.device)
                labels         = labels.to(self.device)
                lengths        = lengths.to(self.device)

                loss, _ = self.model(
                    input_ids, attention_mask, token_type_ids,
                    labels=labels, lengths=lengths,
                )
                loss = loss / GRADIENT_ACCUMULATION_STEPS
                loss.backward()

                if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    torch.nn.utils.clip_grad_norm_(
                        self.model.parameters(), GRAD_CLIP
                    )
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()

                running_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS
                n_steps += 1

            # Final partial accumulation
            if n_steps % GRADIENT_ACCUMULATION_STEPS != 0:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

            epoch_train_time = time.time() - epoch_start   # ← NEW
            avg_train_loss   = running_loss / max(1, n_steps)

            # ── VALIDATION ────────────────────────────────────
            val_loss    = self.compute_val_loss(dev_dataset)
            val_metrics = self.evaluate(dev_dataset, rare_ids)

            print(
                f"Epoch {epoch:02d}/{num_epochs} | "
                f"train_loss: {avg_train_loss:.4f} | "
                f"val_loss: {val_loss:.4f} | "
                f"val_macro_f1: {val_metrics['macro_f1']:.4f} | "
                f"val_rare_f1: {val_metrics['rare_f1']:.4f} | "
                f"val_acc: {val_metrics['accuracy']:.4f} | "
                f"epoch_time: {epoch_train_time:.1f}s"   # ← NEW
            )

            row = {
                "epoch":              epoch,
                "train_loss":         avg_train_loss,
                "val_loss":           val_loss,
                "val_accuracy":       val_metrics["accuracy"],
                # F1
                "val_macro_f1":       val_metrics["macro_f1"],
                "val_micro_f1":       val_metrics["micro_f1"],
                "val_weighted_f1":    val_metrics["weighted_f1"],
                "val_rare_f1":        val_metrics["rare_f1"],
                # Precision
                "val_macro_precision":    val_metrics["macro_precision"],
                "val_micro_precision":    val_metrics["micro_precision"],
                "val_weighted_precision": val_metrics["weighted_precision"],
                "val_rare_precision":     val_metrics["rare_precision"],
                # Recall
                "val_macro_recall":    val_metrics["macro_recall"],
                "val_micro_recall":    val_metrics["micro_recall"],
                "val_weighted_recall": val_metrics["weighted_recall"],
                "val_rare_recall":     val_metrics["rare_recall"],
                # Time  ← NEW
                "epoch_train_time_s":  epoch_train_time,
                "timestamp":           datetime.utcnow().isoformat(),
            }
            history.append(row)

            # ── CHECKPOINT (best macro-F1) ─────────────────────
            if val_metrics["macro_f1"] > best_f1 + 1e-4:
                best_f1    = val_metrics["macro_f1"]
                best_state = {k: v.cpu().clone()
                              for k, v in self.model.state_dict().items()}
                print(f"  ✔ New best val_macro_f1={best_f1:.4f} — snapshot saved")

        # ── Total training time  ← NEW ────────────────────────
        total_train_time = time.time() - total_train_start
        print(f"\n⏱  Total training time : {total_train_time/60:.2f} min "
              f"({total_train_time:.1f} s)")

        # ── SAVE HISTORY & CURVES ─────────────────────────────
        hist_df = pd.DataFrame(history)
        # Append total training time as a metadata row footer
        meta = pd.DataFrame([{
            "epoch": "TOTAL",
            "epoch_train_time_s": total_train_time,
        }])
        hist_df.to_csv(os.path.join(OUT_DIR, "history.csv"), index=False)
        self._plot_history(hist_df)

        # Save timing summary  ← NEW
        timing_summary = {
            "total_training_time_s":   total_train_time,
            "total_training_time_min": total_train_time / 60,
            "avg_epoch_time_s":        total_train_time / num_epochs,
            "num_epochs":              num_epochs,
        }
        with open(os.path.join(OUT_DIR, "timing_summary.json"), "w") as f:
            json.dump(timing_summary, f, indent=2)

        # ── SAVE BEST MODEL ───────────────────────────────────
        if best_state is not None:
            self._save_model_hf(best_state, tokenizer)

        return hist_df, total_train_time

    # ── Evaluation + inference timing  ← NEW ─────────────────
    def evaluate(self, dataset, rare_ids, split_name="dev",
                 measure_inference_time=False):
        """
        Run inference on `dataset`.

        If measure_inference_time=True:
          - records wall-clock time for the full pass
          - records per-sample latency (total / n_samples)
          - records per-sentence latency (total / n_sentences)
          - saves inference_time_<split_name>.json to OUT_DIR
        """
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        n_samples = len(dataset)

        infer_start = time.time() if measure_inference_time else None

        with torch.no_grad():
            for input_ids, attention_mask, token_type_ids, labels, lengths in loader:
                input_ids      = input_ids.to(self.device)
                attention_mask = attention_mask.to(self.device)
                token_type_ids = token_type_ids.to(self.device)
                lengths        = lengths.to(self.device)

                decoded, _ = self.model(
                    input_ids, attention_mask, token_type_ids,
                    labels=None, lengths=lengths,
                )
                for i, seq_preds in enumerate(decoded):
                    true_len = int(lengths[i].item())
                    true_seq = labels[i, :true_len].cpu().numpy().tolist()
                    all_preds.extend(seq_preds)
                    all_trues.extend(true_seq)

        if measure_inference_time:
            total_infer_time = time.time() - infer_start
            n_sentences      = len(all_trues)
            infer_info = {
                "split":                       split_name,
                "n_documents":                 n_samples,
                "n_sentences":                 n_sentences,
                "total_inference_time_s":      total_infer_time,
                "latency_per_document_ms":     total_infer_time / max(1, n_samples) * 1000,
                "latency_per_sentence_ms":     total_infer_time / max(1, n_sentences) * 1000,
                "throughput_sentences_per_s":  n_sentences / max(1e-9, total_infer_time),
            }
            with open(
                os.path.join(OUT_DIR, f"inference_time_{split_name}.json"), "w"
            ) as f:
                json.dump(infer_info, f, indent=2)

            print(f"\n⏱  Inference time ({split_name}): "
                  f"{total_infer_time:.2f} s | "
                  f"latency/doc: {infer_info['latency_per_document_ms']:.1f} ms | "
                  f"latency/sent: {infer_info['latency_per_sentence_ms']:.2f} ms | "
                  f"throughput: {infer_info['throughput_sentences_per_s']:.1f} sent/s")
        else:
            infer_info = None

        metrics = compute_all_metrics(all_trues, all_preds, rare_ids, split_name)
        if infer_info:
            metrics["inference_time_info"] = infer_info
        return metrics

    # ── Save in HuggingFace format ────────────────────────────
    def _save_model_hf(self, state_dict, tokenizer):
        self.model.bert.config.save_pretrained(BEST_MODEL_DIR)
        tokenizer.save_pretrained(BEST_MODEL_DIR)
        torch.save(
            state_dict,
            os.path.join(BEST_MODEL_DIR, "pytorch_model.bin"),
        )
        model_args = {
            "bert_model_name": INLEGALBERT_MODEL_NAME,
            "lstm_hidden":     LSTM_HIDDEN,
            "lstm_layers":     LSTM_LAYERS,
            "num_labels":      NUM_LABELS,
            "dropout":         DROPOUT,
            "labels":          LABELS,
            "label2id":        label2id,
            "id2label":        id2label,
            "max_seq_length":  MAX_SEQ_LENGTH,
            "rare_threshold":  RARE_THRESHOLD,
        }
        with open(os.path.join(BEST_MODEL_DIR, "model_args.json"), "w") as f:
            json.dump(model_args, f, indent=2)

        print(f"\n💾 Best model saved to: {BEST_MODEL_DIR}/")
        print(f"   ├── pytorch_model.bin")
        print(f"   ├── config.json")
        print(f"   ├── tokenizer.json")
        print(f"   ├── vocab.txt")
        print(f"   └── model_args.json")

    # ── Training curves ───────────────────────────────────────
    @staticmethod
    def _plot_history(hist_df):
        """
        Saves four figures:
          training_loss_curve.png   — train loss vs val loss
          training_f1_curve.png     — macro / micro / weighted / rare F1
          training_pr_curve.png     — precision & recall (macro + rare)
          training_time_curve.png   — per-epoch training time  ← NEW
        """
        epochs = hist_df["epoch"].tolist()

        # ── Figure 1: Loss ────────────────────────────────────
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.plot(epochs, hist_df["train_loss"], label="Train Loss",  marker="o")
        ax.plot(epochs, hist_df["val_loss"],   label="Val Loss",    marker="s")
        ax.set_title("Training vs Validation Loss (CRF NLL)")
        ax.set_xlabel("Epoch"); ax.set_ylabel("CRF NLL Loss")
        ax.legend(); ax.grid(True, alpha=0.3)
        plt.tight_layout()
        p = os.path.join(OUT_DIR, "training_loss_curve.png")
        plt.savefig(p, dpi=150); plt.close()
        print(f"Saved {p}")

        # ── Figure 2: F1 metrics ──────────────────────────────
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        for col, lbl, ls in [
            ("val_macro_f1",    "Macro-F1",    "-"),
            ("val_micro_f1",    "Micro-F1",    "--"),
            ("val_weighted_f1", "Weighted-F1", "-."),
            ("val_rare_f1",     "Rare-F1",     ":"),
        ]:
            axes[0].plot(epochs, hist_df[col], label=lbl, linestyle=ls, marker="o")
        axes[0].set_title("Validation F1 Scores")
        axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("F1")
        axes[0].legend(); axes[0].grid(True, alpha=0.3)

        axes[1].plot(epochs, hist_df["train_loss"], label="Train Loss", marker="o")
        axes[1].plot(epochs, hist_df["val_loss"],   label="Val Loss",   marker="s")
        axes[1].set_title("Loss Curves")
        axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Loss")
        axes[1].legend(); axes[1].grid(True, alpha=0.3)

        plt.tight_layout()
        p = os.path.join(OUT_DIR, "training_f1_curve.png")
        plt.savefig(p, dpi=150); plt.close()
        print(f"Saved {p}")

        # ── Figure 3: Precision & Recall ──────────────────────
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        for col, lbl, ls in [
            ("val_macro_precision",    "Macro-Prec",    "-"),
            ("val_micro_precision",    "Micro-Prec",    "--"),
            ("val_weighted_precision", "Weighted-Prec", "-."),
            ("val_rare_precision",     "Rare-Prec",     ":"),
        ]:
            axes[0].plot(epochs, hist_df[col], label=lbl, linestyle=ls, marker="o")
        axes[0].set_title("Validation Precision")
        axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Precision")
        axes[0].legend(); axes[0].grid(True, alpha=0.3)

        for col, lbl, ls in [
            ("val_macro_recall",    "Macro-Rec",    "-"),
            ("val_micro_recall",    "Micro-Rec",    "--"),
            ("val_weighted_recall", "Weighted-Rec", "-."),
            ("val_rare_recall",     "Rare-Rec",     ":"),
        ]:
            axes[1].plot(epochs, hist_df[col], label=lbl, linestyle=ls, marker="o")
        axes[1].set_title("Validation Recall")
        axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Recall")
        axes[1].legend(); axes[1].grid(True, alpha=0.3)

        plt.tight_layout()
        p = os.path.join(OUT_DIR, "training_pr_curve.png")
        plt.savefig(p, dpi=150); plt.close()
        print(f"Saved {p}")

        # ── Figure 4: Per-epoch training time  ← NEW ─────────
        if "epoch_train_time_s" in hist_df.columns:
            fig, ax = plt.subplots(figsize=(8, 4))
            ax.bar(epochs, hist_df["epoch_train_time_s"], color="steelblue", alpha=0.8)
            ax.axhline(
                hist_df["epoch_train_time_s"].mean(),
                color="red", linestyle="--",
                label=f"Mean = {hist_df['epoch_train_time_s'].mean():.1f}s"
            )
            ax.set_title("Per-Epoch Training Time")
            ax.set_xlabel("Epoch"); ax.set_ylabel("Time (s)")
            ax.legend(); ax.grid(True, alpha=0.3, axis="y")
            plt.tight_layout()
            p = os.path.join(OUT_DIR, "training_time_curve.png")
            plt.savefig(p, dpi=150); plt.close()
            print(f"Saved {p}")

    # ── Confusion matrix heatmap ──────────────────────────────
    @staticmethod
    def save_confusion_matrix(cm, split_name, rare_labels=None):
        fig, ax = plt.subplots(figsize=(14, 11))
        sns.heatmap(
            cm, annot=True, fmt="d",
            xticklabels=LABELS, yticklabels=LABELS,
            cmap="Blues", ax=ax,
        )
        if rare_labels:
            for tick in ax.get_xticklabels():
                if tick.get_text() in rare_labels:
                    tick.set_color("red")
            for tick in ax.get_yticklabels():
                if tick.get_text() in rare_labels:
                    tick.set_color("red")

        ax.set_title(
            f"{split_name.capitalize()} Confusion Matrix"
            + (f"\n(red labels = rare ≤ {RARE_THRESHOLD*100:.0f}%)" if rare_labels else "")
        )
        plt.tight_layout()
        path = os.path.join(OUT_DIR, f"{split_name}_confusion_matrix.png")
        plt.savefig(path, dpi=150); plt.close()
        print(f"Saved {path}")

    # ── Per-class F1 bar chart  ← NEW ────────────────────────
    @staticmethod
    def save_per_class_f1_chart(per_class_metrics, split_name, rare_labels=None):
        """
        Horizontal bar chart: per-class F1 scores.
        Rare-class bars are coloured red, common-class bars blue.
        """
        labels = LABELS
        f1s    = [per_class_metrics[l]["f1"] for l in labels]
        colors = ["tomato" if (rare_labels and l in rare_labels) else "steelblue"
                  for l in labels]

        fig, ax = plt.subplots(figsize=(9, 6))
        bars = ax.barh(labels, f1s, color=colors, edgecolor="white")
        ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
        ax.set_xlim(0, 1.12)
        ax.set_xlabel("F1 Score")
        ax.set_title(
            f"{split_name.capitalize()} Per-Class F1"
            + (f"\n(red = rare ≤ {RARE_THRESHOLD*100:.0f}%)" if rare_labels else "")
        )
        ax.grid(True, alpha=0.3, axis="x")
        plt.tight_layout()
        path = os.path.join(OUT_DIR, f"{split_name}_per_class_f1.png")
        plt.savefig(path, dpi=150); plt.close()
        print(f"Saved {path}")


# ═══════════════════════════════════════════════════════════
# PRINT FULL METRICS TABLE
# ═══════════════════════════════════════════════════════════
def print_metrics_table(dev_metrics, test_metrics, total_train_time=None,
                        total_trainable=None, total_frozen=None):
    rows = [
        ("Accuracy",           "accuracy"),
        ("Macro-F1",           "macro_f1"),
        ("Micro-F1",           "micro_f1"),
        ("Weighted-F1",        "weighted_f1"),
        ("Rare / Minority F1", "rare_f1"),
        ("Macro-Precision",    "macro_precision"),
        ("Micro-Precision",    "micro_precision"),
        ("Weighted-Precision", "weighted_precision"),
        ("Rare-Precision",     "rare_precision"),
        ("Macro-Recall",       "macro_recall"),
        ("Micro-Recall",       "micro_recall"),
        ("Weighted-Recall",    "weighted_recall"),
        ("Rare-Recall",        "rare_recall"),
    ]
    print("\n" + "=" * 64)
    print("FINAL RESULTS SUMMARY")
    print("=" * 64)

    # Model / timing info
    if total_trainable is not None:
        print(f"  Trainable Parameters : {total_trainable:,}")
        print(f"  Frozen Parameters    : {total_frozen:,}")
    if total_train_time is not None:
        print(f"  Total Training Time  : {total_train_time/60:.2f} min")
    print("-" * 64)
    print(f"  {'Metric':<28} {'Dev':>12} {'Test':>12}")
    print("-" * 64)

    for label, key in rows:
        sep = "─" * 64 if key == "rare_f1" else ""
        if sep: print(sep)
        print(f"  {label:<28} {dev_metrics[key]:>12.4f} {test_metrics[key]:>12.4f}")
    print("=" * 64)

    # Per-class breakdown  ← NEW
    print("\n  PER-CLASS F1 / PRECISION / RECALL")
    print("  " + "-" * 62)
    print(f"  {'Label':<20} {'F1-Dev':>9} {'F1-Test':>9} "
          f"{'Prec-Test':>11} {'Rec-Test':>10}")
    print("  " + "-" * 62)
    for lbl in LABELS:
        dv = dev_metrics["per_class_metrics"][lbl]
        ts = test_metrics["per_class_metrics"][lbl]
        print(f"  {lbl:<20} {dv['f1']:>9.4f} {ts['f1']:>9.4f} "
              f"{ts['precision']:>11.4f} {ts['recall']:>10.4f}")
    print("  " + "-" * 62)


# ═══════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════
def main():
    print(f"Device: {DEVICE}")

    # ── Load data ─────────────────────────────────────────────
    print("Loading JSONL files...")
    train_raw = load_jsonl(TRAIN_PATH)
    dev_raw   = load_jsonl(DEV_PATH)
    test_raw  = load_jsonl(TEST_PATH)

    train_docs = extract_docs(train_raw)
    dev_docs   = extract_docs(dev_raw)
    test_docs  = extract_docs(test_raw)
    print(f"  Train: {len(train_docs)} | Dev: {len(dev_docs)} | Test: {len(test_docs)}")

    # ── Detect rare classes from TRAIN distribution ───────────
    rare_labels, rare_ids, label_freqs = detect_rare_classes(train_docs)

    freq_df = pd.DataFrame([
        {"label": l, "frequency": label_freqs[l], "is_rare": l in rare_labels}
        for l in LABELS
    ])
    freq_df.to_csv(os.path.join(OUT_DIR, "label_frequencies.csv"), index=False)

    # ── Tokenizer ─────────────────────────────────────────────
    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(INLEGALBERT_MODEL_NAME)

    # ── Datasets ──────────────────────────────────────────────
    train_dataset = RRCDataset(train_docs, tokenizer)
    dev_dataset   = RRCDataset(dev_docs,   tokenizer)
    test_dataset  = RRCDataset(test_docs,  tokenizer)

    # ── Model ─────────────────────────────────────────────────
    print("Initialising InLegalBERT + BiLSTM + CRF ...")
    model = InLegalBERT_BiLSTM_CRF(
        bert_model_name=INLEGALBERT_MODEL_NAME,
        lstm_hidden=LSTM_HIDDEN,
        lstm_layers=LSTM_LAYERS,
        num_labels=NUM_LABELS,
        dropout=DROPOUT,
    )

    # ── Parameter summary  ← NEW ──────────────────────────────
    total_trainable, total_frozen, param_table = count_parameters(model)
    param_df = pd.DataFrame(param_table)
    param_df.to_csv(os.path.join(OUT_DIR, "parameter_summary.csv"), index=False)

    trainer = Trainer(model, device=DEVICE)

    # ── Train ─────────────────────────────────────────────────
    print(f"\nStarting training for {NUM_EPOCHS} epochs...")
    hist_df, total_train_time = trainer.train(   # ← now returns timing too
        train_dataset, dev_dataset,
        rare_ids  =rare_ids,
        tokenizer =tokenizer,
        num_epochs=NUM_EPOCHS,
    )
    print("\nTraining complete.")

    # ── Load best model weights ───────────────────────────────
    best_bin = os.path.join(BEST_MODEL_DIR, "pytorch_model.bin")
    if os.path.exists(best_bin):
        model.load_state_dict(torch.load(best_bin, map_location=DEVICE))
        print("Loaded best checkpoint from pytorch_model.bin")

    # ── Dev evaluation  (with inference timing)  ← NEW ───────
    print("\nEvaluating on Dev set...")
    dev_metrics = trainer.evaluate(
        dev_dataset, rare_ids,
        split_name="dev",
        measure_inference_time=True,
    )
    print(f"  Dev  Accuracy : {dev_metrics['accuracy']:.4f}")
    print(f"  Dev  Macro-F1 : {dev_metrics['macro_f1']:.4f}")
    print(f"  Dev  Rare-F1  : {dev_metrics['rare_f1']:.4f}")

    with open(os.path.join(OUT_DIR, "dev_classification_report.txt"), "w") as f:
        f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n\n")
        f.write(dev_metrics["cls_report"])
    trainer.save_confusion_matrix(dev_metrics["cm"], "dev", rare_labels)
    trainer.save_per_class_f1_chart(               # ← NEW
        dev_metrics["per_class_metrics"], "dev", rare_labels
    )

    # ── Test evaluation  (with inference timing)  ← NEW ──────
    print("\nEvaluating on Test set...")
    test_metrics = trainer.evaluate(
        test_dataset, rare_ids,
        split_name="test",
        measure_inference_time=True,
    )
    print(f"  Test Accuracy : {test_metrics['accuracy']:.4f}")
    print(f"  Test Macro-F1 : {test_metrics['macro_f1']:.4f}")
    print(f"  Test Rare-F1  : {test_metrics['rare_f1']:.4f}")

    with open(os.path.join(OUT_DIR, "test_classification_report.txt"), "w") as f:
        f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n\n")
        f.write(test_metrics["cls_report"])
    trainer.save_confusion_matrix(test_metrics["cm"], "test", rare_labels)
    trainer.save_per_class_f1_chart(               # ← NEW
        test_metrics["per_class_metrics"], "test", rare_labels
    )

    # ── Save predictions ──────────────────────────────────────
    pred_df = pd.DataFrame({
        "true": [id2label[x] for x in test_metrics["all_trues"]],
        "pred": [id2label[x] for x in test_metrics["all_preds"]],
    })
    pred_df.to_csv(os.path.join(OUT_DIR, "test_predictions.csv"), index=False)

    # ── Save per-class metrics as CSV  ← NEW ─────────────────
    for split, mets in [("dev", dev_metrics), ("test", test_metrics)]:
        rows_pc = []
        for lbl in LABELS:
            pc = mets["per_class_metrics"][lbl]
            rows_pc.append({
                "label":     lbl,
                "is_rare":   lbl in rare_labels,
                "f1":        pc["f1"],
                "precision": pc["precision"],
                "recall":    pc["recall"],
            })
        pd.DataFrame(rows_pc).to_csv(
            os.path.join(OUT_DIR, f"{split}_per_class_metrics.csv"), index=False
        )

    # ── Save all metrics as JSON ──────────────────────────────
    scalar_keys = [
        "macro_f1","micro_f1","weighted_f1","rare_f1",
        "macro_precision","micro_precision","weighted_precision","rare_precision",
        "macro_recall","micro_recall","weighted_recall","rare_recall",
        "accuracy",
    ]
    metrics_summary = {
        # Model meta  ← NEW
        "model": {
            "name":               "InLegalBERT + BiLSTM + CRF",
            "bert_model":         INLEGALBERT_MODEL_NAME,
            "trainable_params":   total_trainable,
            "frozen_params":      total_frozen,
            "total_params":       total_trainable + total_frozen,
        },
        # Timing  ← NEW
        "timing": {
            "total_training_time_s":   total_train_time,
            "total_training_time_min": total_train_time / 60,
            "avg_epoch_time_s":        total_train_time / NUM_EPOCHS,
            "dev_inference":  dev_metrics.get("inference_time_info", {}),
            "test_inference": test_metrics.get("inference_time_info", {}),
        },
        "rare_classes":   rare_labels,
        "rare_threshold": RARE_THRESHOLD,
        "dev":  {k: dev_metrics[k]  for k in scalar_keys},
        "test": {k: test_metrics[k] for k in scalar_keys},
        # Per-class  ← NEW
        "per_class_dev":  dev_metrics["per_class_metrics"],
        "per_class_test": test_metrics["per_class_metrics"],
    }
    with open(os.path.join(OUT_DIR, "metrics_summary.json"), "w") as f:
        json.dump(metrics_summary, f, indent=2)

    # ── Final table ───────────────────────────────────────────
    print_metrics_table(
        dev_metrics, test_metrics,
        total_train_time=total_train_time,
        total_trainable=total_trainable,
        total_frozen=total_frozen,
    )

    print(f"\n📁 All outputs saved to: {OUT_DIR}/")
    print("   ├── history.csv                    (per-epoch metrics + epoch time)")
    print("   ├── label_frequencies.csv")
    print("   ├── parameter_summary.csv          ← NEW")
    print("   ├── timing_summary.json            ← NEW")
    print("   ├── training_loss_curve.png")
    print("   ├── training_f1_curve.png")
    print("   ├── training_pr_curve.png")
    print("   ├── training_time_curve.png        ← NEW")
    print("   ├── dev_classification_report.txt")
    print("   ├── dev_confusion_matrix.png")
    print("   ├── dev_per_class_f1.png           ← NEW")
    print("   ├── dev_per_class_metrics.csv      ← NEW")
    print("   ├── inference_time_dev.json        ← NEW")
    print("   ├── test_classification_report.txt")
    print("   ├── test_confusion_matrix.png")
    print("   ├── test_per_class_f1.png          ← NEW")
    print("   ├── test_per_class_metrics.csv     ← NEW")
    print("   ├── inference_time_test.json       ← NEW")
    print("   ├── test_predictions.csv")
    print("   ├── metrics_summary.json           (includes param + timing)")
    print("   └── best_model/")
    print("       ├── pytorch_model.bin")
    print("       ├── config.json")
    print("       ├── tokenizer.json")
    print("       ├── vocab.txt")
    print("       └── model_args.json")


if __name__ == "__main__":
    main()

Device: cuda:0
Loading JSONL files...
  Train: 245 | Dev: 30 | Test: 50

📊 Label frequency analysis (threshold ≤ 5%):
   PREAMBLE             14.50%  ( 4167 samples)
   FAC                  19.99%  ( 5744 samples)
   RLC                   2.62%  (  752 samples) ← RARE
   ISSUE                 1.28%  (  367 samples) ← RARE
   ARG_PETITIONER        4.58%  ( 1315 samples) ← RARE
   ARG_RESPONDENT        2.43%  (  698 samples) ← RARE
   ANALYSIS             36.66%  (10537 samples)
   STA                   1.67%  (  481 samples) ← RARE
   PRE_RELIED            4.97%  ( 1427 samples) ← RARE
   PRE_NOT_RELIED        0.55%  (  158 samples) ← RARE
   RATIO                 2.30%  (  661 samples) ← RARE
   RPC                   3.67%  ( 1055 samples) ← RARE
   NONE                  4.79%  ( 1377 samples) ← RARE

   Rare classes (10): ['RLC', 'ISSUE', 'ARG_PETITIONER', 'ARG_RESPONDENT', 'STA', 'PRE_RELIED', 'PRE_NOT_RELIED', 'RATIO', 'RPC', 'NONE']

Loading tokenizer...
Initialising InLegalBERT + 

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



MODEL PARAMETER SUMMARY
  Component                      Trainable     Frozen        Total
------------------------------------------------------------------
  InLegalBERT Encoder          109,482,240          0  109,482,240
  BiLSTM                           107,008          0      107,008
  Classifier Head                      749          0          749
  CRF                                  195          0          195
  Dropout                                0          0            0
──────────────────────────────────────────────────────────────────
  ── TOTAL ──                  109,590,192          0  109,590,192

Starting training for 60 epochs...
Epoch 01/60 | train_loss: 299.9976 | val_loss: 235.7267 | val_macro_f1: 0.0020 | val_rare_f1: 0.0026 | val_acc: 0.0131 | epoch_time: 62.4s
  ✔ New best val_macro_f1=0.0020 — snapshot saved
Epoch 02/60 | train_loss: 269.1282 | val_loss: 202.6917 | val_macro_f1: 0.0480 | val_rare_f1: 0.0000 | val_acc: 0.3478 | epoch_time: 66.8s
  ✔ New 

In [2]:
!python -m pip install seaborn

In [2]:
!python -m pip install pytorch-crf

